# 03. Serving the fine-tuned model with Ray Serve

**AI Drug Discovery & Development Summit 2026 · Hands-on workshop · Part 3 of 3**

**Workshop series:** [01 Ray and offline inference](01_ray_and_offline_inference.ipynb) · [02 Train and tune](02_train_and_tune.ipynb) · [03 Serve](03_serve.ipynb)

Notebook 02 produced a fine-tuned checkpoint. This notebook puts it behind an endpoint a chemist can use:
the 3D results viewer, prediction from Python, and swapping checkpoints without breaking what is already
in use.

| § | Topic | Ray library | Min |
|---|---|---|---|
| 0 | Setup | | 2 |
| 6 | Serving the fine-tuned model | Ray Serve | 17 |
| 7 | Summary and next steps | | 5 |

Section numbers follow the workshop as a whole, so this notebook starts at Section 6.


## 0. Setup

- The Workspace runs the `aiddd-boltz2-of3` image: Ray, Boltz-2 and 2 OpenFold3 versions (0.5 and 0.4.3).
- This notebook runs on a CPU-only head node. L4 GPU nodes join when a cell requests a GPU and leave when idle.
- Data, model weights and all benchmark results live on shared storage.

In [34]:
import os, sys, json, time, warnings, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import ray

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

from aiddd_pipeline import (SHARED, TARGETS, BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT, OF3_LEGACY,
                            Boltz2Predictor, OF3Predictor, load_ligands, validate, make_scorer)
import workshop_helpers as wh

# Data and weights are downloaded once, ahead of the workshop (README step 2)
needed = [TARGETS[t].path("metadata") for t in wh.TARGET_KEYS] + [BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT]
missing = [str(p) for p in needed if not p.exists()]
assert not missing, f"Missing {missing}. Run `python scripts/setup_data.py` and `python scripts/prepare_mpro.py` first."

# True: run every GPU step live (the presenter, or anyone with their own GPUs).
# False: GPU steps load stored results. Serving is notebook 03.
USE_GPUS = False
SERVICE_URL = os.environ.get("AIDDD_SERVICE_URL", "")   # the deployed prediction service used in notebook 03
GPU = "L4"            # GPU type for every GPU step in this notebook

wh.MINE.mkdir(parents=True, exist_ok=True)
print(f"shared data:  {SHARED}")
print(f"your outputs: {wh.MINE}")

try:
    ray.init(runtime_env={"env_vars": {"AIDDD_SHARED": str(SHARED)}}, ignore_reinit_error=True, logging_level="ERROR")
except ValueError:   # inside an Anyscale Job, which already sets the environment
    ray.init(ignore_reinit_error=True, logging_level="ERROR")
dashboard = os.environ.get("RAY_OVERRIDE_DASHBOARD_URL") or ray.get_runtime_context().dashboard_url
print(f"Ray Dashboard metrics: {dashboard.rstrip('/')}/#/metrics   (open in a new browser tab)")

shared data:  /mnt/user_storage/aiddd
your outputs: /mnt/user_storage/aiddd/workshop/alicia_chua_30e1ffa
Ray Dashboard metrics: https://session-thqdxs9cduqis7hsp7r4mvd42v.i.anyscaleuserdata.com/#/metrics   (open in a new browser tab)


In [ ]:
# Context from notebooks 01 and 02, loaded here so this notebook stands alone
results = {t: wh.load_results(t) for t in wh.TARGET_KEYS}
ev71 = results["ev71-2a"]
x7259a = ev71[ev71.complex_name == "A71EV2A-x7259a"].iloc[0]     # the late follow-on used in 6.3
print("loaded benchmark results:", ", ".join(results))


## 6. Serving with Ray Serve: one base model, many specialists

⏱ **Budget: 17 minutes.**

A chemist does not want a different endpoint per model. **Model multiplexing** serves the general **base
model** and every program's **specialist** from one pool of GPU replicas, chosen per request. This section
runs that pool in the Workspace and walks the lifecycle:

1. **6.1** Start the pool with the base model.
2. **6.2–6.3** Predict with the base model: it is weak on EV-A71 2A.
3. **6.4** Add the program's specialist (notebook 02's fine-tune) and predict again.
4. **6.5** Add the next version and roll it out.

`OF3Pool` in `serve_viewer_multi.py` is the whole mechanism:

```python
@serve.deployment(ray_actor_options={"num_gpus": 0.5})
class OF3Pool:
    @serve.multiplexed(max_num_models_per_replica=3)      # keep 3 checkpoints hot per replica
    async def get_model(self, model_id: str):
        return load_checkpoint(self.registry[model_id])   # loaded on first request for this id

    async def __call__(self, smiles: str):
        model = await self.get_model(serve.get_multiplexed_model_id())
        return model.predict(smiles)
```

### 6.1 Start the pool with the base model

`serve.run` starts the deployment in this Workspace and returns a `DeploymentHandle`. The registry passed
as `user_config` is the set of models the pool can serve; each checkpoint loads on the first request for
its id. We start with only the base model.

In [ ]:
from ray import serve
from serve_viewer_multi import OF3Pool

assert USE_GPUS, "Section 6 runs the pool in this Workspace: set USE_GPUS = True (needs an L4)."

# id -> checkpoint. "legacy" uses the OpenFold3 0.4.3 runtime that the OF3-p2 checkpoints need.
REGISTRY = {
    "base": {"name": "Base model (OF3-p2 145k)", "checkpoint": str(OF3_P2_CKPT), "legacy": True, "samples": 1},
}

def serve_pool(registry):
    """Start or reconfigure the pool with this registry, and return a handle to it."""
    return serve.run(OF3Pool.options(user_config={"models": registry}).bind(),
                     name="aiddd-pose", route_prefix="/pose")

handle = serve_pool(REGISTRY)
print("pool serving:", list(REGISTRY))

### 6.2 Compounds for the demo

All are EV-A71 2A late follow-ons with a crystal structure, so you can compare the prediction with the
experiment.

| Compound | Why it is interesting | SMILES |
|---|---|---|
| **x7101a** | 145k misses by 6.0 Å; a fine-tuned model lands within 0.5 Å | `CSc1ncnc(N2CCC3(CCCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7259a** | Most potent shown here (pKD 7.9) | `CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7259a, S → O** | A new molecule with no crystal structure | `COc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |

### 6.3 Predict with the base model

`predict` names the model with `handle.options(multiplexed_model_id=...)` — the one line that picks a
model per request. The base model has never seen this pocket with a ligand in it, so on an EV-A71 2A
follow-on it is usually wrong.

In [ ]:
from aiddd_pipeline import score_pose
MY_SMILES = "CSc1ncnc(N2CCC3(CCCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"     # x7101a, crystal A71EV2A-x7101a
CRYSTAL = "A71EV2A-x7101a"

def crystal_rmsd(cif_path, complex_name):
    """Ligand RMSD of a predicted pose to that compound's crystal, the same metric as the benchmark."""
    row = results["ev71-2a"].query("complex_name == @complex_name").iloc[0]
    ref_pdb = Path(row.ref_pdb)
    ref_sdf = ref_pdb.with_name(ref_pdb.name.replace("_complex_ref.pdb", "_ligand_ref.sdf"))
    return score_pose(str(cif_path), ref_pdb, ref_sdf, {})[0]

async def predict(smiles, model, complex_name=None):
    out = await handle.options(multiplexed_model_id=model).remote(smiles)   # picks the model per request
    out["lig_rmsd"] = crystal_rmsd(out["cif_path"], complex_name) if complex_name and out.get("cif_path") else None
    r = out["lig_rmsd"]
    print(f"{model:9s} status={out['status']:7s} replica={out.get('replica')}"
          + ("" if r is None else f" ligand_RMSD={r:.2f} Å"))
    return out

out = await predict(MY_SMILES, "base", CRYSTAL)
if out.get("cif_path"):
    wh.show_analog(out["cif_path"], results["ev71-2a"].query("complex_name == @CRYSTAL").iloc[0],
                   label="x7101a (base model)", height=340)

### 6.4 Add the program's specialist and predict again

Notebook 02 fine-tuned the base model on this program's fragment screen. **Adding it is a new entry in the
registry, not a swap**: the base model keeps serving everyone else, and the specialist loads on the first
request for its id. Re-running `serve.run` with the larger registry reconfigures the pool in place.

In [ ]:
from aiddd_pipeline import latest_finetune_ckpt
ev71_v1 = latest_finetune_ckpt() or OF3_FT_CKPT     # notebook 02's checkpoint, or the release as a fallback

REGISTRY["ev71-v1"] = {"name": "EV-A71 specialist v1 (notebook 02)", "checkpoint": str(ev71_v1),
                       "legacy": True, "samples": 1}
handle = serve_pool(REGISTRY)                        # reconfigure: base + ev71-v1
print("pool serving:", list(REGISTRY))

out = await predict(MY_SMILES, "ev71-v1", CRYSTAL)   # loads on first request, then stays hot
if out.get("cif_path"):
    wh.show_analog(out["cif_path"], results["ev71-2a"].query("complex_name == 'A71EV2A-x7101a'").iloc[0],
                   label="x7101a (specialist v1)", height=340)

The same molecule, same endpoint, a different `multiplexed_model_id`: the predicted pose moves from
~6 Å off the crystal to within ~0.5 Å. The base model and the specialist share the same GPU replica.

### 6.5 Add the next version and roll it out

A stronger specialist arrives (here, the OpenBind release). Add it as a new id; the old one stays
callable, so you can compare them, promote the new one, or reproduce an old pose for an audit.

In [ ]:
REGISTRY["ev71-v2"] = {"name": "EV-A71 specialist v2 (OpenBind release)", "checkpoint": str(OF3_FT_CKPT),
                       "legacy": True, "samples": 1}
handle = serve_pool(REGISTRY)                        # reconfigure: base + ev71-v1 + ev71-v2
print("pool serving:", list(REGISTRY))

for model in ["ev71-v1", "ev71-v2"]:                 # both versions live, pick per request
    await predict(MY_SMILES, model, CRYSTAL)

**Two ways to roll out, both without downtime.**

- **Within the pool (per request).** Point a program at a new id; the old id stays loaded, so rollback is
  instant and any past pose stays reproducible. A new molecule takes 1–3 minutes; a repeat is cached.
- **Across service versions (share of traffic).** An Anyscale Service runs a new version next to the
  current one and shifts traffic gradually:

```bash
anyscale service deploy -f jobs/service_multi.yaml --canary-percent 10     # new version takes 10%
anyscale service deploy -f jobs/service_multi.yaml --canary-percent 100    # promote
anyscale service rollback -n aiddd-viewer-multi                      # or roll back
```

| What Ray Serve and Anyscale add | Why it matters |
|---|---|
| **Model multiplexing** | The base model and every specialist on the same GPUs, chosen per request |
| **On-demand loading** | A new model is served by requesting its id; no redeploy |
| **GPUs from every zone** | When one zone runs out of L4s, replicas start in another |
| **Composition in Python** | `serve_viewer_multi.py` also composes Boltz-2 affinity and the web viewer around the pool |
| **Canary rollouts and rollback** | A new model reaches a share of users first |

## 7. Summary and next steps

**Findings**

- **Generalization.** Models that reach 88 to 92% on SARS-CoV-2 late follow-ons fall to 1 to 24% on EV-A71 2A.
- **Fine-tuning.** 66 fragment-screen structures measurably improve OpenFold3 on held-out follow-on
  compounds (notebook 02), and the result is a specialist for that target.

**Delivered on a single Ray cluster**

- **Ray Data.** The full benchmark across 3 campaigns, and 402 new OpenFold3 predictions in 19 minutes on
  12 L4 GPUs.
- **Ray Train.** OpenFold3's own training code on 8 GPUs, with checkpoints and automatic recovery.
- **Ray Serve.** The base model and program specialists behind one multiplexed endpoint, chosen per
  request, with on-demand loading and canary rollouts.

The [README](README.md) has the scientific background and the full runbook.

Run the next cell to stop the pool and release the GPU nodes.

In [ ]:
serve.shutdown()
print("Pool stopped. Idle GPU nodes will be released within a few minutes.")